# 06 — Model Improvement

## 1. Objective

The goal of this notebook is to improve the current best model, XGBoost, based on the error analysis findings.

We will test:
- class imbalance handling;
- probability threshold adjustment;
- hyperparameter tuning.

Model improvements will be evaluated mainly using Macro F1 and Class 1 Recall.

## 2. Rebuild the Current XGBoost Model

Before testing improvements, we rebuild the current XGBoost model using the same train-validation split.

This provides a consistent reference for comparing new experiments.

In [1]:
import sys
sys.path.append("..")

from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, recall_score
from xgboost import XGBClassifier

from src.data_split import load_and_split_data
from src.preprocessing import build_tree_preprocessor

X_train, X_valid, y_train, y_valid = load_and_split_data()

baseline_xgb = Pipeline(
    steps=[
        ("preprocessor", build_tree_preprocessor()),
        ("model", XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            random_state=42,
            eval_metric="logloss"
        ))
    ]
)

baseline_xgb.fit(X_train, y_train)

baseline_pred = baseline_xgb.predict(X_valid)

print("Accuracy:", round(accuracy_score(y_valid, baseline_pred), 4))
print("Macro F1:", round(f1_score(y_valid, baseline_pred, average="macro"), 4))
print("Class 1 Recall:", round(recall_score(y_valid, baseline_pred), 4))

Accuracy: 0.8918
Macro F1: 0.7104
Class 1 Recall: 0.3565


## 3. Handle Class Imbalance

Because the positive class is much smaller than the negative class, we test class weighting using `scale_pos_weight`.

The weight is calculated from the training data as:

`number of negative samples / number of positive samples`

In [2]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("scale_pos_weight:", round(scale_pos_weight, 2))

scale_pos_weight: 6.1
